In [ ]:

!nvidia-smi

Thu Jul  9 12:34:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content
!git clone https://github.com/princeton-vl/RAFT-Stereo.git
%cd RAFT-Stereo

/content
Cloning into 'RAFT-Stereo'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 87 (delta 40), reused 33 (delta 33), pack-reused 29 (from 1)
Receiving objects: 100% (87/87), 703.12 KiB | 16.35 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/RAFT-Stereo


In [ ]:
!pip install torch torchvision torchaudio
!pip install opencv-python numpy matplotlib pillow scipy imageio tqdm tensorboard opt_einsum einops scikit-image pyyaml tifffile imagecodecs

In [ ]:
!mkdir -p models
!cp /content/drive/MyDrive/stereo_raft_colab/raftstereo-eth3d.pth models/
!mkdir -p /content/raft_out

In [ ]:
%cd /content/RAFT-Stereo

!python demo.py \
  --restore_ckpt models/raftstereo-eth3d.pth \
  -l /content/drive/MyDrive/stereo_raft_colab/rectified_left.png \
  -r /content/drive/MyDrive/stereo_raft_colab/rectified_right.png \
  --save_numpy \
  --corr_implementation alt \
  --valid_iters 8

/content/RAFT-Stereo
Traceback (most recent call last):
  File "/content/RAFT-Stereo/demo.py", line 78, in <module>
    demo(args)
  File "/content/RAFT-Stereo/demo.py", line 25, in demo
    model.load_state_dict(torch.load(args.restore_ckpt))
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'models/raftstereo-eth3d.pth'


In [ ]:
!ls -lh /content/drive/MyDrive/stereo_raft_colab

total 187M
-rw------- 1 root root 1.9K Jul  8 03:52 convert_raft_to_pipeline_depth.py
-rw------- 1 root root 9.0K Jul  8 04:35 evaluate_depth.py
-rw------- 1 root root  98M Jul  8 04:06 gt_depth_rectified_float.tiff
-rw------- 1 root root  43M Jul  8 02:06 raftstereo-eth3d.pth
-rw------- 1 root root  23M Jul  8 04:03 rectified_left.png
-rw------- 1 root root  23M Jul  8 04:03 rectified_right.png


In [ ]:
%cd /content/RAFT-Stereo
!mkdir -p models
!cp /content/drive/MyDrive/stereo_raft_colab/raftstereo-eth3d.pth models/
!ls -lh models

/content/RAFT-Stereo
total 43M
-rw------- 1 root root 43M Jul  9 12:42 raftstereo-eth3d.pth


In [ ]:
!python demo.py \
  --restore_ckpt models/raftstereo-eth3d.pth \
  -l /content/drive/MyDrive/stereo_raft_colab/rectified_left.png \
  -r /content/drive/MyDrive/stereo_raft_colab/rectified_right.png \
  --save_numpy \
  --corr_implementation alt \
  --valid_iters 8

Found 1 images. Saving files to demo_output/
  0% 0/1 [00:00<?, ?it/s]/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
  0% 0/1 [00:06<?, ?it/s]
Traceback (most recent call last):
  File "/content/RAFT-Stereo/demo.py", line 78, in <module>
    demo(args)
  File "/content/RAFT-Stereo/demo.py", line 46, in demo
    _, flow_up = model(image1, image2, iters=args.valid_iters, test_mode=True)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import cv2
from pathlib import Path

left_path = "/content/drive/MyDrive/stereo_raft_colab/rectified_left.png"
right_path = "/content/drive/MyDrive/stereo_raft_colab/rectified_right.png"

out_dir = Path("/content/raft_out")
out_dir.mkdir(parents=True, exist_ok=True)

left = cv2.imread(left_path)
right = cv2.imread(right_path)

target_width = 2048
scale = target_width / left.shape[1]
target_height = int(left.shape[0] * scale)

target_width = (target_width // 32) * 32
target_height = (target_height // 32) * 32

left_small = cv2.resize(left, (target_width, target_height), interpolation=cv2.INTER_AREA)
right_small = cv2.resize(right, (target_width, target_height), interpolation=cv2.INTER_AREA)

cv2.imwrite("/content/raft_out/left_2048.png", left_small)
cv2.imwrite("/content/raft_out/right_2048.png", right_small)

print("Original:", left.shape[1], "x", left.shape[0])
print("Resized:", target_width, "x", target_height)

Original: 6208 x 4135
Resized: 2048 x 1344


In [ ]:
%cd /content/RAFT-Stereo

!rm -rf demo_output
!python demo.py \
  --restore_ckpt models/raftstereo-eth3d.pth \
  -l /content/raft_out/left_2048.png \
  -r /content/raft_out/right_2048.png \
  --save_numpy \
  --corr_implementation alt \
  --valid_iters 8 \
  --mixed_precision

/content/RAFT-Stereo
Found 1 images. Saving files to demo_output/
  0% 0/1 [00:00<?, ?it/s]/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT-Stereo/core/raft_stereo.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
100% 1/1 [00:04<00:00,  4.67s/it]


In [ ]:
!ls -lh demo_output

total 11M
-rw-r--r-- 1 root root  11M Jul  9 12:45 raft_out.npy
-rw-r--r-- 1 root root 247K Jul  9 12:45 raft_out.png


In [ ]:
!cp /content/RAFT-Stereo/demo_output/raft_out.npy /content/drive/MyDrive/stereo_raft_colab/disparity_raft_2048.npy
!cp /content/RAFT-Stereo/demo_output/raft_out.png /content/drive/MyDrive/stereo_raft_colab/disparity_raft_2048.png

In [ ]:
!ls -lh /content/drive/MyDrive/stereo_raft_colab

total 197M
-rw------- 1 root root 1.9K Jul  8 03:52 convert_raft_to_pipeline_depth.py
-rw------- 1 root root  11M Jul  9 12:46 disparity_raft_2048.npy
-rw------- 1 root root 247K Jul  9 12:46 disparity_raft_2048.png
-rw------- 1 root root 9.0K Jul  8 04:35 evaluate_depth.py
-rw------- 1 root root  98M Jul  8 04:06 gt_depth_rectified_float.tiff
-rw------- 1 root root  43M Jul  8 02:06 raftstereo-eth3d.pth
-rw------- 1 root root  23M Jul  8 04:03 rectified_left.png
-rw------- 1 root root  23M Jul  8 04:03 rectified_right.png


In [ ]:
!mkdir -p /content/out
!mkdir -p /content/scripts

!cp /content/drive/MyDrive/stereo_raft_colab/gt_depth_rectified_float.tiff /content/out/
!cp /content/drive/MyDrive/stereo_raft_colab/evaluate_depth.py /content/scripts/

In [ ]:
from pathlib import Path
import numpy as np
import tifffile

OUT_DIR = Path("/content/out")
RAFT_DISP_PATH = Path("/content/drive/MyDrive/stereo_raft_colab/disparity_raft_2048.npy")

DEPTH_OUT = OUT_DIR / "depth_raft_float.tiff"
MASK_OUT = OUT_DIR / "valid_depth_raft_mask.tiff"
DISP_OUT = OUT_DIR / "disparity_raft_float.tiff"

FX_FULL = 3408.59
CAMERA_WIDTH_FULL = 6208
BASELINE = 0.4289

disparity = np.load(str(RAFT_DISP_PATH)).astype(np.float32)
disparity = np.squeeze(disparity)
disparity = np.abs(disparity)

h, w = disparity.shape
fx_scaled = FX_FULL * (w / CAMERA_WIDTH_FULL)

valid = np.isfinite(disparity) & (disparity > 1e-6)

depth = np.full((h, w), np.nan, dtype=np.float32)
depth[valid] = (fx_scaled * BASELINE) / disparity[valid]

mask = np.zeros((h, w), dtype=np.uint8)
mask[valid & np.isfinite(depth) & (depth > 0)] = 255

tifffile.imwrite(str(DEPTH_OUT), depth.astype(np.float32))
tifffile.imwrite(str(MASK_OUT), mask)
tifffile.imwrite(str(DISP_OUT), disparity.astype(np.float32))

print("Converted RAFT 2048 output")
print("Resolution:", w, "x", h)
print("Scaled fx:", fx_scaled)
print("Wrote:", DEPTH_OUT)
print("Wrote:", MASK_OUT)

Converted RAFT 2048 output
Resolution: 2048 x 1344
Scaled fx: 1124.4832989690722
Wrote: /content/out/depth_raft_float.tiff
Wrote: /content/out/valid_depth_raft_mask.tiff


In [ ]:
!pip install tifffile imagecodecs opencv-python matplotlib

In [ ]:
!python /content/scripts/evaluate_depth.py --out /content/out --tags raft

Loaded GT: /content/out/gt_depth_rectified_float.tiff (928,177 valid px)

Depth evaluation vs. rectified ETH3D ground truth:
                           raft
  -----------------------------
  valid px               99,210
  coverage               100.0%
  MAE (m)                2.0974
  RMSE (m)               2.3612
  median AE (m)          1.8735
  MAE scaled (m)         0.0587
  scale (gt/pred)        1.3186
  bad > 0.1m             100.0%
  bad > 0.5m             100.0%
  bad > 1.0m              79.9%

Error maps:
  wrote error_raft_scaled.png
